# YOLOv11m Segmentation 추가학습 파이프라인 (로컬 GPU)

## 사전 준비
1. GitHub clone:
   ```
   git clone https://github.com/Samingyeong/CCATFARM.git
   ```
2. `data/` 폴더를 `CCATFARM/Model-jetson/` 안에 생성 후 아래 파일 넣기:
   - `data/kkat_dataset/` (gofile에서 다운)
   - `data/ccat/` (CVAT export - image/annotations.xml + 이미지들)
   - `data/yolo11m_seg_aug_20260707_155305_best.pt` (모델 파일)

## 셀 순서대로 실행

## 1. GPU 확인 + 패키지 설치

In [ ]:
!nvidia-smi
!pip install ultralytics albumentations -q

## 2. 경로 설정

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트 (이 노트북이 있는 위치 기준)
PROJECT_ROOT = Path('.').resolve()
print(f'프로젝트 루트: {PROJECT_ROOT}')

# 경로 설정
SCRIPTS_DIR = PROJECT_ROOT  # 스크립트 파일들 위치
DATA_DIR = PROJECT_ROOT / 'data'  # 원본 데이터 위치
DATASET_ALL = PROJECT_ROOT / 'dataset_all'  # 출력 데이터
RUNS_DIR = PROJECT_ROOT / 'runs'  # 학습 결과

CCAT_DIR = DATA_DIR / 'ccat'
KKAT_DIR = DATA_DIR / 'kkat_dataset'
MODEL_FILE = DATA_DIR / 'yolo11m_seg_aug_20260707_155305_best.pt'

# 확인
print(f'\nccat 폴더: {CCAT_DIR} -> 존재: {CCAT_DIR.exists()}')
print(f'kkat 폴더: {KKAT_DIR} -> 존재: {KKAT_DIR.exists()}')
print(f'모델 파일: {MODEL_FILE} -> 존재: {MODEL_FILE.exists()}')

## 3. 데이터 합치기 (01_merge_dataset.py)
- ccat (CVAT XML) → YOLO seg 포맷 변환
- kkat_dataset + 변환된 ccat → dataset_all/original에 합침

In [ ]:
!python 01_merge_dataset.py \
  --ccat "{CCAT_DIR}" \
  --kkat "{KKAT_DIR}" \
  --output "{DATASET_ALL}"

## 4. 1차 추가학습 (03_train_step1.py)
- 155305 모델 + ccat 24장(원본)으로 fine-tuning
- → 1차 모델 생성

In [ ]:
!python 03_train_step1.py \
  --model "{MODEL_FILE}" \
  --dataset "{DATASET_ALL}" \
  --output "{RUNS_DIR}"

## 5. 데이터 증강 (02_augment_dataset.py)
- 원본 전체 5배 증강 (Albumentations)
- train/val 분할 + data.yaml 생성

In [ ]:
!python 02_augment_dataset.py \
  --dataset "{DATASET_ALL}"

## 6. 2차 추가학습 (03_train_step2.py)
- 1차 모델 + 전체 데이터(원본+증강)로 학습
- → 최종 모델 생성

In [ ]:
!python 03_train_step2.py \
  --model "{RUNS_DIR}/step1_ccat_finetune/weights/best.pt" \
  --dataset "{DATASET_ALL}" \
  --output "{RUNS_DIR}"

## 7. 테스트 (04_test.py)

In [ ]:
!python 04_test.py \
  --model "{RUNS_DIR}/step2_full_finetune/weights/best.pt" \
  --dataset "{DATASET_ALL}" \
  --output "{RUNS_DIR}"

## 8. 결과 시각화

In [ ]:
from IPython.display import Image, display
import glob

# 학습 곡선
results_img = str(RUNS_DIR / 'step2_full_finetune' / 'results.png')
if os.path.exists(results_img):
    print('=== 학습 곡선 ===')
    display(Image(results_img, width=800))

# 예측 결과
pred_images = sorted(glob.glob(str(RUNS_DIR / 'test_results' / '*.jpg')))[:6]
for img_path in pred_images:
    print(os.path.basename(img_path))
    display(Image(img_path, width=600))

## 9. 최종 모델 저장 확인

In [ ]:
best_model = RUNS_DIR / 'step2_full_finetune' / 'weights' / 'best.pt'
step1_model = RUNS_DIR / 'step1_ccat_finetune' / 'weights' / 'best.pt'

print(f'1차 모델: {step1_model}')
print(f'  존재: {step1_model.exists()}')
print(f'\n최종 모델: {best_model}')
print(f'  존재: {best_model.exists()}')
print(f'\n이 파일을 Jetson에 배포하면 됨.')
print('ONNX/TensorRT 변환은 별도 진행.')